In [1]:
!pip install seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=57a6a1972740930c0a1b52c908bb37ebb40209ebea0e080f066221ef9f29fbcb
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [2]:
# dependencies
import torch
import torch.optim as optim
from transformers import BertForTokenClassification, BertTokenizerFast
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report
from seqeval.metrics import classification_report as seqeval_report
from seqeval.metrics import f1_score as seqeval_f1
from seqeval.metrics import precision_score as seqeval_precision
from seqeval.metrics import recall_score as seqeval_recall
from tqdm.auto import tqdm  # modern, works in notebooks & scripts

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("alaakhaled/conll003-englishversion")

print("Path to dataset files:", path)

base_path = path + "/"

Using Colab cache for faster access to the 'conll003-englishversion' dataset.
Path to dataset files: /kaggle/input/conll003-englishversion


In [4]:
# read the data files
def load_sentences(filepath):

    sentences = []
    tokens = []
    pos_tags = []
    chunk_tags = []
    ner_tags = []

    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f.readlines():
            # sentence boundary
            if (line.startswith('-DOCSTART-') or line.strip() == ''):
                if len(tokens) > 0:
                    sentences.append({
                        'tokens': tokens,
                        'pos_tags': pos_tags,
                        'chunk_tags': chunk_tags,
                        'ner_tags': ner_tags
                    })
                    tokens = []
                    pos_tags = []
                    chunk_tags = []
                    ner_tags = []
            else:
                l = line.strip().split(' ')
                if len(l) >= 4:
                    tokens.append(l[0])
                    pos_tags.append(l[1])
                    chunk_tags.append(l[2])
                    ner_tags.append(l[3])
    # last sentence if file doesn't end with blank line
    if len(tokens) > 0:
        sentences.append({
            'tokens': tokens,
            'pos_tags': pos_tags,
            'chunk_tags': chunk_tags,
            'ner_tags': ner_tags
        })
    return sentences

print('loading data')
train_sentences = load_sentences(base_path + 'train.txt')
test_sentences = load_sentences(base_path + 'test.txt')
valid_sentences = load_sentences(base_path + 'valid.txt')

loading data


In [5]:
train_sentences[0:1]

[{'tokens': ['EU',
   'rejects',
   'German',
   'call',
   'to',
   'boycott',
   'British',
   'lamb',
   '.'],
  'pos_tags': ['NNP', 'VBZ', 'JJ', 'NN', 'TO', 'VB', 'JJ', 'NN', '.'],
  'chunk_tags': ['B-NP',
   'B-VP',
   'B-NP',
   'I-NP',
   'B-VP',
   'I-VP',
   'B-NP',
   'I-NP',
   'O'],
  'ner_tags': ['B-ORG', 'O', 'B-MISC', 'O', 'O', 'O', 'B-MISC', 'O', 'O']}]

In [5]:
# build tag set and label mappings
all_tags = sorted({tag for s in train_sentences for tag in s['ner_tags']})
label2id = {tag: i for i, tag in enumerate(all_tags)}
id2label = {i: tag for tag, i in label2id.items()}
num_labels = len(all_tags)
print('Tagset size:', num_labels)
print('Tags:', all_tags)
print(f"Tag IDs: {id2label}")
print(f"Label2ID: {label2id}")

Tagset size: 9
Tags: ['B-LOC', 'B-MISC', 'B-ORG', 'B-PER', 'I-LOC', 'I-MISC', 'I-ORG', 'I-PER', 'O']
Tag IDs: {0: 'B-LOC', 1: 'B-MISC', 2: 'B-ORG', 3: 'B-PER', 4: 'I-LOC', 5: 'I-MISC', 6: 'I-ORG', 7: 'I-PER', 8: 'O'}
Label2ID: {'B-LOC': 0, 'B-MISC': 1, 'B-ORG': 2, 'B-PER': 3, 'I-LOC': 4, 'I-MISC': 5, 'I-ORG': 6, 'I-PER': 7, 'O': 8}


In [7]:
# load BERT tokenizer
bert_version = 'bert-base-uncased'
tokenizer = BertTokenizerFast.from_pretrained(bert_version)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [8]:
tokenizer

BertTokenizer(name_or_path='bert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [12]:
# %%capture
# map tokens and tags to token ids and label ids
def align_label(tokens, labels, sentence):
    """
    It maps:
      - O means the word doesn’t correspond to any entity.
      - B-PER/I-PER means the word corresponds to the beginning of/is inside a person entity.
      - B-ORG/I-ORG means the word corresponds to the beginning of/is inside an organization entity.
      - B-LOC/I-LOC means the word corresponds to the beginning of/is inside a location entity.
      - B-MISC/I-MISC means the word corresponds to the beginning of/is inside a miscellaneous entity.
    to each token in the sentence.

    When it runs out of tokens, it adds -100 to the label. Each word index that does not correspond to a word
    in the sentence, gets a -100. Each unknown token gets a -100.

    INPUTS:

    OUTPUT:
      - labels: 512 length of labels for each token in the sentence.
      - tokens: tokenizer output (BatchEncoding) with word_ids()
      - labels: list of original word-level labels for a single sentence

    INFO:
    512: is the model output size.
    """
    word_ids = tokens.word_ids()
    # print(f"Word IDS: {word_ids}")
    previous_word_idx = None
    label_ids = []
    # print(tokens)

    for word_idx in word_ids:
        # print(f"\n===\n\tword idx: {word_idx}")

        # Debug
        # if word_idx is not None:
        #     print(f"Sentence token: {sentence["tokens"][word_idx]}")
        #     print(f"Sentence label: {sentence["ner_tags"][word_idx]}")

        if word_idx is None:
            # special tokens
            label_ids.append(-100)
            # print(f"Special token: {label_ids}")

        elif word_idx != previous_word_idx:
            # first subword of a word
            # Get the corresponding token NER tag from the labels or -100
            label_ids.append(label2id.get(labels[word_idx], -100))
            # print(f"First subword: {label_ids}")

        else:
            # subsequent subword of the same word (ignore for loss)
            label_ids.append(-100)
            # print(f"Subsequent subword: {label_ids}")

        previous_word_idx = word_idx

    # print(f"Word IDX: {word_idx}, labels: {label_ids}")

    return label_ids

def encode(sentence):
    encodings = tokenizer(
        sentence['tokens'],
        truncation=True,
        padding='max_length',
        is_split_into_words=True,
        return_tensors='pt'  # get tensors directly
    )
    # print(f"Encodings: {encodings}")
    labels = align_label(encodings, sentence['ner_tags'], sentence)
    # print(f"Final labels: {labels}")

    return {
        'input_ids': encodings['input_ids'].squeeze(0),        # [seq_len]
        'attention_mask': encodings['attention_mask'].squeeze(0),
        'labels': torch.tensor(labels, dtype=torch.long)
    }

print('encoding data')
# for sentence in train_sentences:
#     print(f"Sentence: {sentence}")
#     encode(sentence)
#     break
train_dataset = [encode(sentence) for sentence in train_sentences]
valid_dataset = [encode(sentence) for sentence in valid_sentences]
test_dataset = [encode(sentence) for sentence in test_sentences]

encoding data


In [10]:
train_dataset[0]

{'input_ids': tensor([  101,  7327, 19164,  2446,  2655,  2000, 17757,  2329, 12559,  1012,
           102,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,   

In [ ]:
torch.nn.functional.cross_entropy(
    # torch.tensor([15.], dtype=torch.long, requires_grad=True).float(),
    torch.randn(1, 1, requires_grad=True),
    torch.empty(1, dtype=torch.long).random_(1)
    # torch.tensor([1])
    )

tensor(0., grad_fn=<NllLossBackward0>)

In [ ]:
input = torch.randn(1, 2, requires_grad=True)
# target = torch.empty(1, dtype=torch.long).random_(2)
target = torch.tensor([1], dtype=torch.long)

output = torch.nn.functional.cross_entropy(input, target)
print(f"Input: {input}")
print(f"target: {target}")
print(f"Output: {output}")

Input: tensor([[0.1930, 0.5774]], requires_grad=True)
target: tensor([1])
Output: 0.5192708969116211


In [13]:
# PyTorch Dataset wrapper for better compatibility with DataLoader
class InputDataset(torch.utils.data.Dataset):
    def __init__(self, data):
        self.data = data
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx]

train_dataset = InputDataset(train_dataset)
valid_dataset = InputDataset(valid_dataset)
test_dataset = InputDataset(test_dataset)

In [12]:
# hyper-parameters
EPOCHS = 3
BATCH_SIZE = 8
LR = 1e-5

# use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# initialize the model including a classification layer with num_labels classes
print('initializing the model')
model = BertForTokenClassification.from_pretrained(
    bert_version,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)
model.to(device)
optimizer = optim.AdamW(params=model.parameters(), lr=LR)

# prepare batches of data
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)
valid_loader = torch.utils.data.DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE
)
test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE
)

initializing the model


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized be

In [16]:
# evaluate the performance of the model
def EvaluateModel(model, data_loader):
    """
    Returns:
      Y_actual_flat, Y_preds_flat : 1D numpy arrays of label ids over all
        evaluated tokens (special/subword tokens excluded). Used for token-level
        accuracy, balanced accuracy and the sklearn classification report.
      y_true_tags, y_pred_tags    : lists of lists of BIO tag strings, one
        sub-list per sentence. Used for entity-level evaluation with seqeval.
    """
    model.eval()
    Y_actual_flat, Y_preds_flat = [], []
    y_true_tags, y_pred_tags = [], []

    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Evaluating"):
            # move the batch tensors to the same device as the model
            batch = {k: v.to(device) for k, v in batch.items()}
            # send 'input_ids', 'attention_mask' and 'labels' to the model
            outputs = model(**batch)
            logits = outputs.logits
            preds = torch.argmax(logits, dim=-1)  # [batch, seq_len]

            # iterate through the examples in the batch
            for idx in range(batch['labels'].size(0)):
                true_values_all = batch['labels'][idx]
                mask = (true_values_all != -100)

                true_values = true_values_all[mask]
                pred_values = preds[idx][mask]

                # accumulate flat tensors for token-level metrics
                Y_actual_flat.append(true_values)
                Y_preds_flat.append(pred_values)

                # accumulate per-sentence BIO tag strings for seqeval
                true_tags_sent = [id2label[i] for i in true_values.tolist()]
                pred_tags_sent = [id2label[i] for i in pred_values.tolist()]
                y_true_tags.append(true_tags_sent)
                y_pred_tags.append(pred_tags_sent)

    Y_actual_flat = torch.cat(Y_actual_flat).detach().cpu().numpy()
    Y_preds_flat = torch.cat(Y_preds_flat).detach().cpu().numpy()

    return Y_actual_flat, Y_preds_flat, y_true_tags, y_pred_tags


def report_metrics(Y_actual, Y_preds, y_true_tags, y_pred_tags, split_name):
    """Print both token-level and entity-level metrics."""
    print(f"\n=== {split_name} — Token-level metrics ===")
    print("Accuracy          : {:.3f}".format(accuracy_score(Y_actual, Y_preds)))
    print("Balanced accuracy : {:.3f}".format(balanced_accuracy_score(Y_actual, Y_preds)))

    print(f"\n=== {split_name} — Entity-level metrics (seqeval) ===")
    print("Precision : {:.3f}".format(seqeval_precision(y_true_tags, y_pred_tags)))
    print("Recall    : {:.3f}".format(seqeval_recall(y_true_tags, y_pred_tags)))
    print("F1        : {:.3f}".format(seqeval_f1(y_true_tags, y_pred_tags)))




In [14]:
# train the model - SKIP if model loaded
print('training the model')
for epoch in range(EPOCHS):
    model.train()
    print(f'epoch {epoch + 1}/{EPOCHS}')
    for batch in tqdm(train_loader, desc=f"Training epoch {epoch + 1}"):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # calculate performance on validation set
    Y_actual, Y_preds, y_true_tags, y_pred_tags = EvaluateModel(model, valid_loader)
    report_metrics(Y_actual, Y_preds, y_true_tags, y_pred_tags,
                   split_name=f"Validation (epoch {epoch + 1})")


training the model
epoch 1/3


Training epoch 1:   0%|          | 0/1756 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [15]:
from google.colab import drive


drive.mount('/content/gdrive')
model_path = "/content/gdrive/My Drive/Colab Notebooks/saved_models"

Mounted at /content/gdrive


In [ ]:
#### Save the model


# Create if path does not exist in google drive
!mkdir -p "/content/gdrive/My Drive/Colab Notebooks/saved_models"

# Save pre-trained model in path with name nlp_ass_3
model.save_pretrained(model_path)

Mounted at /content/gdrive


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [16]:
# Load model from google drive
model = BertForTokenClassification.from_pretrained(
    model_path,
    # num_labels=num_labels
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

# We need to align these predictions with the original words, skipping special tokens
model.to(device)

Loading weights:   0%|          | 0/199 [00:01<?, ?it/s]

BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, el

In [17]:
print('\napplying the model to the test set')
Y_actual, Y_preds, y_true_tags, y_pred_tags = EvaluateModel(model, test_loader)

report_metrics(Y_actual, Y_preds, y_true_tags, y_pred_tags, split_name="Test")


applying the model to the test set


Evaluating:   0%|          | 0/432 [00:00<?, ?it/s]


=== Test — Token-level metrics ===
Accuracy          : 0.980
Balanced accuracy : 0.908

=== Test — Entity-level metrics (seqeval) ===
Precision : 0.897
Recall    : 0.909
F1        : 0.903


In [18]:
# detailed token-level classification report (per-tag, including 'O')
label_ids_sorted = list(range(num_labels))
target_names = [id2label[i] for i in label_ids_sorted]
print("\n=== Test — Token-level classification report (sklearn) ===")
print(
    classification_report(
        Y_actual,
        Y_preds,
        labels=label_ids_sorted,
        target_names=target_names,
        zero_division=0
    )
)


=== Test — Token-level classification report (sklearn) ===
              precision    recall  f1-score   support

       B-LOC       0.94      0.92      0.93      1668
      B-MISC       0.83      0.83      0.83       702
       B-ORG       0.90      0.91      0.90      1661
       B-PER       0.98      0.97      0.97      1617
       I-LOC       0.81      0.89      0.85       257
      I-MISC       0.65      0.77      0.71       216
       I-ORG       0.86      0.89      0.88       835
       I-PER       0.98      0.99      0.99      1156
           O       0.99      0.99      0.99     38323

    accuracy                           0.98     46435
   macro avg       0.88      0.91      0.89     46435
weighted avg       0.98      0.98      0.98     46435



## Task 1

In [19]:
# detailed entity-level classification report (per entity type, no 'O')
print("=== Test — Entity-level classification report (seqeval) ===")
print(
    seqeval_report(
        y_true_tags,
        y_pred_tags,
        digits=3,
        zero_division=0
    )
)

=== Test — Entity-level classification report (seqeval) ===
              precision    recall  f1-score   support

         LOC      0.915     0.919     0.917      1668
        MISC      0.772     0.805     0.788       702
         ORG      0.862     0.886     0.874      1661
         PER      0.971     0.968     0.969      1617

   micro avg      0.897     0.909     0.903      5648
   macro avg      0.880     0.894     0.887      5648
weighted avg      0.898     0.909     0.903      5648



---

In [ ]:
# Find a test sentence where the model fails to tag properly
for i in range(len(test_sentences)):
    sentence_tokens = test_sentences[i]['tokens']
    true_tags = y_true_tags[i]
    predicted_tags = y_pred_tags[i]

    if len(sentence_tokens) >= 13 and true_tags != predicted_tags:
        print(f"Sentence: {' '.join(sentence_tokens)}")
        print(f"True Tags: {true_tags}")
        print(f"Predicted Tags: {predicted_tags}")
        break

Sentence: Cuttitta announced his retirement after the 1995 World Cup , where he took issue with being dropped from the Italy side that faced England in the pool stages .
True Tags: ['B-PER', 'O', 'O', 'O', 'O', 'O', 'B-MISC', 'I-MISC', 'I-MISC', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-LOC', 'O', 'O', 'O', 'B-LOC', 'O', 'O', 'O', 'O', 'O']
Predicted Tags: ['B-PER', 'O', 'O', 'O', 'O', 'O', 'O', 'B-MISC', 'I-MISC', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-LOC', 'O', 'O', 'O', 'B-LOC', 'O', 'O', 'O', 'O', 'O']


In [ ]:
matching_idx = []
missed_idx = []
for i in range(len(true_tags)):
    tt = true_tags[i]
    pt = predicted_tags[i]
    if tt == pt:
        matching_idx.append(i)
    else:
        missed_idx.append(i)

print("Matched NER tokens:")
print(list(map(lambda i: sentence_tokens[i], matching_idx)))
# print(sentence_tokens[matching_idx])

print("Missed NER tokens:")
print(list(map(lambda i: sentence_tokens[i], missed_idx)))
# print(sentence_tokens[missed_idx])

Matched NER tokens:
['Cuttitta', 'announced', 'his', 'retirement', 'after', 'the', 'Cup', ',', 'where', 'he', 'took', 'issue', 'with', 'being', 'dropped', 'from', 'the', 'Italy', 'side', 'that', 'faced', 'England', 'in', 'the', 'pool', 'stages', '.']
Missed NER tokens:
['1995', 'World']


In [ ]:
classified_tokens_info = {
    "correctly_classified": [],
    "incorrectly_classified": []
}

for i in matching_idx:
    classified_tokens_info["correctly_classified"].append(
        (sentence_tokens[i], true_tags[i], predicted_tags[i])
    )

for i in missed_idx:
    classified_tokens_info["incorrectly_classified"].append(
        (sentence_tokens[i], true_tags[i], predicted_tags[i])
    )

display(classified_tokens_info)

{'correctly_classified': [('Cuttitta', 'B-PER', 'B-PER'),
  ('announced', 'O', 'O'),
  ('his', 'O', 'O'),
  ('retirement', 'O', 'O'),
  ('after', 'O', 'O'),
  ('the', 'O', 'O'),
  ('Cup', 'I-MISC', 'I-MISC'),
  (',', 'O', 'O'),
  ('where', 'O', 'O'),
  ('he', 'O', 'O'),
  ('took', 'O', 'O'),
  ('issue', 'O', 'O'),
  ('with', 'O', 'O'),
  ('being', 'O', 'O'),
  ('dropped', 'O', 'O'),
  ('from', 'O', 'O'),
  ('the', 'O', 'O'),
  ('Italy', 'B-LOC', 'B-LOC'),
  ('side', 'O', 'O'),
  ('that', 'O', 'O'),
  ('faced', 'O', 'O'),
  ('England', 'B-LOC', 'B-LOC'),
  ('in', 'O', 'O'),
  ('the', 'O', 'O'),
  ('pool', 'O', 'O'),
  ('stages', 'O', 'O'),
  ('.', 'O', 'O')],
 'incorrectly_classified': [('1995', 'B-MISC', 'O'),
  ('World', 'I-MISC', 'B-MISC')]}

In [ ]:
custom_test_sentence = """
Tim Cook to become
Apple Executive Chairman

John Ternus
to become Apple CEO
"""


# Tokenize
inputs_batch_encoding = tokenizer(
    custom_test_sentence.split(), # Split into words
    is_split_into_words=True,
    return_tensors='pt',
    padding='max_length',
    truncation=True
)

# Extract word_ids
word_ids = inputs_batch_encoding.word_ids()

# Move to device
inputs = {k: v.to(device) for k, v in inputs_batch_encoding.items()}

# Get preds
model.eval()
with torch.no_grad():
    outputs = model(**inputs)

# Get the predicted label IDs
logits = outputs.logits
predictions = torch.argmax(logits, dim=-1).squeeze().tolist()

# Map the predicted IDs to labels
aligned_predictions = []
for i, pred_id in enumerate(predictions):
    if word_ids[i] is not None and (word_ids[i] != word_ids[i-1] if i > 0 else True): # Only consider first token subword
        aligned_predictions.append(id2label[pred_id])
    elif word_ids[i] is None:
        continue # Skip specials: [CLS], [SEP], [PAD]
    else:
        continue # Skip subsequent subwords I-*


print("Original Sentence:", custom_test_sentence)
print("Predicted Tags:", aligned_predictions)

Original Sentence: 
Tim Cook to become
Apple Executive Chairman

John Ternus
to become Apple CEO

Predicted Tags: ['B-PER', 'I-PER', 'O', 'O', 'B-ORG', 'O', 'O', 'B-PER', 'I-PER', 'O', 'O', 'B-ORG', 'O']


In [ ]:
# Prettier print

custom_sentence_words = custom_test_sentence.split()
predicted_classifications = []

min_len = min(len(custom_sentence_words), len(aligned_predictions))

for i in range(min_len):
    predicted_classifications.append(
        (custom_sentence_words[i], aligned_predictions[i])
    )

print("Predicted classifications for custom sentence:")
for word, tag in predicted_classifications:
    print(f"  Word: '{word}' -> Predicted Tag: '{tag}'")

Predicted classifications for custom sentence:
  Word: 'Tim' -> Predicted Tag: 'B-PER'
  Word: 'Cook' -> Predicted Tag: 'I-PER'
  Word: 'to' -> Predicted Tag: 'O'
  Word: 'become' -> Predicted Tag: 'O'
  Word: 'Apple' -> Predicted Tag: 'B-ORG'
  Word: 'Executive' -> Predicted Tag: 'O'
  Word: 'Chairman' -> Predicted Tag: 'O'
  Word: 'John' -> Predicted Tag: 'B-PER'
  Word: 'Ternus' -> Predicted Tag: 'I-PER'
  Word: 'to' -> Predicted Tag: 'O'
  Word: 'become' -> Predicted Tag: 'O'
  Word: 'Apple' -> Predicted Tag: 'B-ORG'
  Word: 'CEO' -> Predicted Tag: 'O'


---

## Task 5

Τροποποιήστε τον αρχικό κώδικα και επαναλάβετε το ερώτημα 1 **έχοντας παγώσει τα βάρη που
αφορούν το προ-εκπαιδευμένο γλωσσικό μοντέλο BERT**, οπότε στη φάση της εκπαίδευσης θα
καθοριστούν **μόνο τα βάρη του επιπέδου ταξινόμησης**.

Αναφέρετε ποιες αλλαγές έγιναν στον
κώδικα.

Αναφέρετε πόσες παράμετροι του μοντέλου έχουν γίνει freeze και πόσες όχι.

Πώς
συγκρίνονται τα αποτελέσματα με αυτά του ερωτήματος 1;

In [ ]:
# Pull model from online as is
print("Pulling model")
model = BertForTokenClassification.from_pretrained(
    bert_version,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)
model.to(device)

# Freeze base model weights, except classification head
for param in model.bert.parameters():
    param.requires_grad = False

Pulling model


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized be

In [ ]:
model

BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, el

In [ ]:
# Count trainable and non-trainable parameters
def count_model_params(model):
    nof_trainable = 0
    nof_frozen = 0
    for param in model.parameters():
        if param.requires_grad == True:
            nof_trainable += sum(p.numel() for p in param)
        elif param.requires_grad == False:
            nof_frozen += sum(p.numel() for p in param)
        else:
            print("Something went wrong!")
            break

    print(f"Trainable params: {nof_trainable}")
    print(f"Frozen params: {nof_frozen}")

In [ ]:
# Get parameter counts
count_model_params(model)

Trainable params: 6921
Frozen params: 108891648


In [ ]:
# Training function
def train_model(model, train_loader, valid_loader):
    """ Will train only the classification head """
    EPOCHS = 3
    BATCH_SIZE = 8
    LR = 1e-5
    optimizer = optim.AdamW(filter(
        lambda p: p.requires_grad, model.parameters()), lr=LR)

    # train the model - SKIP if model loaded
    print('training the model')
    for epoch in range(EPOCHS):
        model.train()
        print(f'epoch {epoch + 1}/{EPOCHS}')
        for batch in tqdm(train_loader, desc=f"Training epoch {epoch + 1}"):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # calculate performance on validation set
        Y_actual, Y_preds, y_true_tags, y_pred_tags = EvaluateModel(model, valid_loader)
        report_metrics(Y_actual, Y_preds, y_true_tags, y_pred_tags,
                      split_name=f"Validation (epoch {epoch + 1})")

    return Y_actual, Y_preds, y_true_tags, y_pred_tags

In [ ]:
Y_actual, Y_preds, y_true_tags, y_pred_tags = train_model(model, train_loader, valid_loader)

training the model
epoch 1/3


Training epoch 1:   0%|          | 0/1756 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/407 [00:00<?, ?it/s]


=== Validation (epoch 1) — Token-level metrics ===
Accuracy          : 0.837
Balanced accuracy : 0.133

=== Validation (epoch 1) — Entity-level metrics (seqeval) ===
Precision : 0.388
Recall    : 0.039
F1        : 0.070
epoch 2/3


Training epoch 2:   0%|          | 0/1756 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/407 [00:00<?, ?it/s]


=== Validation (epoch 2) — Token-level metrics ===
Accuracy          : 0.855
Balanced accuracy : 0.194

=== Validation (epoch 2) — Entity-level metrics (seqeval) ===
Precision : 0.502
Recall    : 0.137
F1        : 0.215
epoch 3/3


Training epoch 3:   0%|          | 0/1756 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/407 [00:00<?, ?it/s]


=== Validation (epoch 3) — Token-level metrics ===
Accuracy          : 0.883
Balanced accuracy : 0.292

=== Validation (epoch 3) — Entity-level metrics (seqeval) ===
Precision : 0.557
Recall    : 0.290
F1        : 0.381


In [19]:
# Print full model evaluation for task 1 table

def full_eval(Y_actual, Y_preds, y_true_tags, y_pred_tags):
    # detailed token-level classification report (per-tag, including 'O')
    label_ids_sorted = list(range(num_labels))
    target_names = [id2label[i] for i in label_ids_sorted]
    print("\n=== Test — Token-level classification report (sklearn) ===")
    print(
        classification_report(
            Y_actual,
            Y_preds,
            labels=label_ids_sorted,
            target_names=target_names,
            zero_division=0
        )
    )

    # detailed entity-level classification report (per entity type, no 'O')
    print("=== Test — Entity-level classification report (seqeval) ===")
    print(
        seqeval_report(
            y_true_tags,
            y_pred_tags,
            digits=3,
            zero_division=0
        )
    )

In [ ]:
full_eval(Y_actual, Y_preds, y_true_tags, y_pred_tags)


=== Test — Token-level classification report (sklearn) ===
              precision    recall  f1-score   support

       B-LOC       0.70      0.55      0.61      1837
      B-MISC       0.82      0.01      0.02       922
       B-ORG       0.70      0.30      0.42      1341
       B-PER       0.88      0.42      0.57      1842
       I-LOC       0.00      0.00      0.00       257
      I-MISC       0.00      0.00      0.00       346
       I-ORG       1.00      0.00      0.00       751
       I-PER       0.94      0.35      0.51      1307
           O       0.89      1.00      0.94     42759

    accuracy                           0.88     51362
   macro avg       0.66      0.29      0.34     51362
weighted avg       0.87      0.88      0.85     51362

=== Test — Entity-level classification report (seqeval) ===
              precision    recall  f1-score   support

         LOC      0.639     0.505     0.564      1837
        MISC      0.727     0.009     0.017       922
         ORG

We can see that keeping the weights frozen, results in poorer classification performance for the specific dataset.

---

## Tasks 6

Τροποποιήστε τον αρχικό κώδικα ώστε αντί για αναγνώριση ονοματικών οντοτήτων να πραγματοποιείται επισημείωση μέρους-του-Λόγου (POS tagging).

Αναφέρετε ποιες αλλαγές έγιναν στον κώδικα.

Επαναλάβετε τα ερωτήματα 1 και 3 για τον τροποποιημένο κώδικα.

In [ ]:
# build tag set and label mappings
all_tags = sorted({tag for s in train_sentences for tag in s["pos_tags"]}) # ner_tags --> pos_tags
label2id = {tag: i for i, tag in enumerate(all_tags)}
id2label = {i: tag for tag, i in label2id.items()}
num_labels = len(all_tags)
print('Tagset size:', num_labels)
print('Tags:', all_tags)
print(f"Tag IDs: {id2label}")
print(f"Label2ID: {label2id}")

Tagset size: 45
Tags: ['"', '$', "''", '(', ')', ',', '.', ':', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'LS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'NN|SYM', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB']
Tag IDs: {0: '"', 1: '$', 2: "''", 3: '(', 4: ')', 5: ',', 6: '.', 7: ':', 8: 'CC', 9: 'CD', 10: 'DT', 11: 'EX', 12: 'FW', 13: 'IN', 14: 'JJ', 15: 'JJR', 16: 'JJS', 17: 'LS', 18: 'MD', 19: 'NN', 20: 'NNP', 21: 'NNPS', 22: 'NNS', 23: 'NN|SYM', 24: 'PDT', 25: 'POS', 26: 'PRP', 27: 'PRP$', 28: 'RB', 29: 'RBR', 30: 'RBS', 31: 'RP', 32: 'SYM', 33: 'TO', 34: 'UH', 35: 'VB', 36: 'VBD', 37: 'VBG', 38: 'VBN', 39: 'VBP', 40: 'VBZ', 41: 'WDT', 42: 'WP', 43: 'WP$', 44: 'WRB'}
Label2ID: {'"': 0, '$': 1, "''": 2, '(': 3, ')': 4, ',': 5, '.': 6, ':': 7, 'CC': 8, 'CD': 9, 'DT': 10, 'EX': 11, 'FW': 12, 'IN': 13, 'JJ': 14, 'JJR': 15, 'JJS': 16, 'LS': 17, 'MD': 18, 'NN': 19, 'NNP': 20, 'NNPS': 21, 

In [ ]:
# map tokens and tags to token ids and label ids
def align_label(tokens, labels, sentence):
    """
    It maps:
      - O means the word doesn’t correspond to any entity.
      - B-PER/I-PER means the word corresponds to the beginning of/is inside a person entity.
      - B-ORG/I-ORG means the word corresponds to the beginning of/is inside an organization entity.
      - B-LOC/I-LOC means the word corresponds to the beginning of/is inside a location entity.
      - B-MISC/I-MISC means the word corresponds to the beginning of/is inside a miscellaneous entity.
    to each token in the sentence.

    When it runs out of tokens, it adds -100 to the label. Each word index that does not correspond to a word
    in the sentence, gets a -100. Each unknown token gets a -100.

    INPUTS:

    OUTPUT:
      - labels: 512 length of labels for each token in the sentence.
      - tokens: tokenizer output (BatchEncoding) with word_ids()
      - labels: list of original word-level labels for a single sentence

    INFO:
    512: is the model output size.
    """
    word_ids = tokens.word_ids()
    # print(f"Word IDS: {word_ids}")
    previous_word_idx = None
    label_ids = []
    # print(tokens)

    for word_idx in word_ids:
        # print(f"\n===\n\tword idx: {word_idx}")

        # Debug
        # if word_idx is not None:
        #     print(f"Sentence token: {sentence["tokens"][word_idx]}")
        #     print(f"Sentence label: {sentence["ner_tags"][word_idx]}")

        if word_idx is None:
            # special tokens
            label_ids.append(-100)
            # print(f"Special token: {label_ids}")

        elif word_idx != previous_word_idx:
            # first subword of a word
            # Get the corresponding token NER tag from the labels or -100
            label_ids.append(label2id.get(labels[word_idx], -100))
            # print(f"First subword: {label_ids}")

        else:
            # subsequent subword of the same word (ignore for loss)
            label_ids.append(-100)
            # print(f"Subsequent subword: {label_ids}")

        previous_word_idx = word_idx

    # print(f"Word IDX: {word_idx}, labels: {label_ids}")

    return label_ids

def encode(sentence):
    encodings = tokenizer(
        sentence['tokens'],
        truncation=True,
        padding='max_length',
        is_split_into_words=True,
        return_tensors='pt'  # get tensors directly
    )
    # print(f"Encodings: {encodings}")
    labels = align_label(encodings, sentence['pos_tags'], sentence) # ner_tags -> pos_tags
    # print(f"Final labels: {labels}")

    return {
        'input_ids': encodings['input_ids'].squeeze(0),        # [seq_len]
        'attention_mask': encodings['attention_mask'].squeeze(0),
        'labels': torch.tensor(labels, dtype=torch.long)
    }

print('encoding data')
# for sentence in train_sentences:
#     print(f"Sentence: {sentence}")
#     encode(sentence)
#     break
train_dataset = [encode(sentence) for sentence in train_sentences]
valid_dataset = [encode(sentence) for sentence in valid_sentences]
test_dataset = [encode(sentence) for sentence in test_sentences]

encoding data


In [ ]:
Y_actual, Y_preds, y_true_tags, y_pred_tags = train_model(model, train_loader, valid_loader)

training the model
epoch 1/3


Training epoch 1:   0%|          | 0/1756 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/407 [00:00<?, ?it/s]


=== Validation (epoch 1) — Token-level metrics ===
Accuracy          : 0.944
Balanced accuracy : 0.811

=== Validation (epoch 1) — Entity-level metrics (seqeval) ===


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NNP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: : seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: IN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: . seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarni

...

Training epoch 3:   0%|          | 0/1756 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/407 [00:00<?, ?it/s]


=== Validation (epoch 3) — Token-level metrics ===
Accuracy          : 0.949
Balanced accuracy : 0.841

=== Validation (epoch 3) — Entity-level metrics (seqeval) ===


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NNP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: : seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: IN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: . seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarni

Precision : 0.926
Recall    : 0.925
F1        : 0.926


In [ ]:
full_eval(Y_actual, Y_preds, y_true_tags, y_pred_tags)


=== Test — Token-level classification report (sklearn) ===
              precision    recall  f1-score   support

           "       1.00      1.00      1.00       640
           $       0.97      1.00      0.99       101
          ''       1.00      0.45      0.62        11
           (       1.00      1.00      1.00       679
           )       1.00      1.00      1.00       680
           ,       1.00      1.00      1.00      1949
           .       1.00      1.00      1.00      1879
           :       1.00      1.00      1.00       624
          CC       1.00      1.00      1.00       932
          CD       0.95      0.99      0.97      4296
          DT       1.00      0.99      0.99      3521
          EX       0.93      1.00      0.96        40
          FW       0.61      0.38      0.47        29
          IN       0.99      0.98      0.98      4977
          JJ       0.89      0.85      0.87      3043
         JJR       0.86      0.83      0.84       105
         JJS       0.

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NNP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: : seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: IN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: . seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarni

              precision    recall  f1-score   support

           '      1.000     0.455     0.625        11
           B      0.892     0.893     0.892      1907
          BD      0.949     0.950     0.949      2224
          BG      0.915     0.927     0.921       699
          BN      0.875     0.849     0.862       928
          BP      0.900     0.838     0.868       365
          BR      0.729     0.660     0.693        53
          BS      0.882     0.833     0.857        18
          BZ      0.929     0.925     0.927       509
           C      1.000     0.999     0.999       932
           D      0.931     0.974     0.952      3231
          DT      0.919     0.914     0.916       162
           H      0.000     0.000     0.000         5
           J      0.859     0.825     0.841      2765
          JR      0.861     0.829     0.845       105
          JS      0.933     0.897     0.915        78
           N      0.906     0.903     0.905      8147
          NP      0.865    

In [ ]:
def find_the_errors_in_pos_tags(my_model, all_test_sentences, actual_tags_from_eval, predicted_tags_from_eval, my_tokenizer, my_tag_id_map, current_device):
    """Identifies and displays the first misclassified sentence (over 13 tokens).
    """
    for sentence_idx in range(len(all_test_sentences)):
        original_words_in_sentence = all_test_sentences[sentence_idx]['tokens']
        true_aligned_pos_tags = actual_tags_from_eval[sentence_idx]
        model_predicted_pos_tags = predicted_tags_from_eval[sentence_idx]

        if not true_aligned_pos_tags: # Skip empty aligned tags
            continue

        # Capture sentences > 13 tokens
        if len(original_words_in_sentence) > 13 and true_aligned_pos_tags != model_predicted_pos_tags:
            print(f"Misclassified sentence (over 13 words long!) at index {sentence_idx}:")
            print(f"Original words: {' '.join(original_words_in_sentence)}")
            print(f"True Tags (aligned): {true_aligned_pos_tags}")
            print(f"Predicted Tags (aligned): {model_predicted_pos_tags}")

            # Re-tokenize to get word_ids for accurate alignment
            tokenized_sentence_for_alignment = my_tokenizer(
                original_words_in_sentence,
                is_split_into_words=True,
                truncation=True,
                padding='max_length',
                return_tensors='pt'
            )
            word_id_mapping_from_tokenizer = tokenized_sentence_for_alignment.word_ids(batch_index=0)

            # Build word list aligned with subword-level tags
            words_for_display = []
            last_word_seen_idx = None
            aligned_tag_list_position = 0

            for token_word_id in word_id_mapping_from_tokenizer:
                if token_word_id is not None and token_word_id != last_word_seen_idx:
                    if aligned_tag_list_position < len(true_aligned_pos_tags):
                        words_for_display.append(original_words_in_sentence[token_word_id])
                        aligned_tag_list_position += 1
                    last_word_seen_idx = token_word_id

            # Compare true vs. predicted tags word by word
            classification_breakdown = {
                "correctly_classified_words": [],
                "incorrectly_classified_words": []
            }

            for word_index_in_display_list in range(len(words_for_display)):
                word = words_for_display[word_index_in_display_list]
                true_tag = true_aligned_pos_tags[word_index_in_display_list]
                predicted_tag = model_predicted_pos_tags[word_index_in_display_list]

                if true_tag == predicted_tag:
                    classification_breakdown["correctly_classified_words"].append(
                        (word, true_tag, predicted_tag)
                    )
                else:
                    classification_breakdown["incorrectly_classified_words"].append(
                        (word, true_tag, predicted_tag)
                    )

            display(classification_breakdown)
            break # Stop after the first instance


In [ ]:
# Find a misclassified sentence - task 3 (now looking for sentences > 13 tokens)
find_the_errors_in_pos_tags(model, test_sentences, y_true_tags, y_pred_tags, tokenizer, id2label, device)

Misclassified sentence (over 13 words long!) at index 4:
Original words: But China saw their luck desert them in the second match of the group , crashing to a surprise 2-0 defeat to newcomers Uzbekistan .
True Tags (aligned): ['IN', 'VBG', 'NNP', 'RP', 'IN', 'CD', 'IN', 'DT', 'NN', 'NN', 'IN', 'NNP', 'NNP', ',', 'NNP', 'VBD', 'PRP$', 'JJ', 'NN', 'IN', 'CD', 'VBZ', 'IN', 'VBG', 'VBD', 'RP', 'IN', 'CD', 'IN', 'NNP', 'VBP', 'NNP', 'NNP', 'VBG', 'CD', 'IN', 'CD', '.']
Predicted Tags (aligned): ['IN', 'NN', 'NNP', 'IN', 'IN', 'CD', 'IN', 'DT', 'NN', 'NN', 'IN', 'NNP', 'NNP', ',', 'NNP', 'VBD', 'PRP$', 'JJ', 'NN', 'IN', 'CD', 'NNS', 'IN', 'VBG', 'VBN', 'RP', 'IN', 'CD', 'IN', 'NNP', 'NN', 'NNP', 'NNP', 'VBG', 'CD', 'IN', 'CD', '.']


{'correctly_classified_words': [('But', 'IN', 'IN'),
  ('saw', 'NNP', 'NNP'),
  ('luck', 'IN', 'IN'),
  ('desert', 'CD', 'CD'),
  ('them', 'IN', 'IN'),
  ('in', 'DT', 'DT'),
  ('the', 'NN', 'NN'),
  ('second', 'NN', 'NN'),
  ('match', 'IN', 'IN'),
  ('of', 'NNP', 'NNP'),
  ('the', 'NNP', 'NNP'),
  ('group', ',', ','),
  (',', 'NNP', 'NNP'),
  ('crashing', 'VBD', 'VBD'),
  ('to', 'PRP$', 'PRP$'),
  ('a', 'JJ', 'JJ'),
  ('surprise', 'NN', 'NN'),
  ('2-0', 'IN', 'IN'),
  ('defeat', 'CD', 'CD'),
  ('newcomers', 'IN', 'IN'),
  ('Uzbekistan', 'VBG', 'VBG')],
 'incorrectly_classified_words': [('China', 'VBG', 'NN'),
  ('their', 'RP', 'IN'),
  ('to', 'VBZ', 'NNS'),
  ('.', 'VBD', 'VBN')]}

In [ ]:
custom_test_sentence = """
Tim Cook to become
Apple Executive Chairman

John Ternus
to become Apple CEO
"""


# Tokenize
inputs_batch_encoding = tokenizer(
    custom_test_sentence.split(), # Split into words
    is_split_into_words=True,
    return_tensors='pt',
    padding='max_length',
    truncation=True
)

# Extract word_ids
word_ids = inputs_batch_encoding.word_ids()

# Move to device
inputs = {k: v.to(device) for k, v in inputs_batch_encoding.items()}

# Get preds
model.eval()
with torch.no_grad():
    outputs = model(**inputs)

# Get the predicted label IDs
logits = outputs.logits
predictions = torch.argmax(logits, dim=-1).squeeze().tolist()

# Map the predicted IDs to labels
aligned_predictions = []
for i, pred_id in enumerate(predictions):
    if word_ids[i] is not None and (word_ids[i] != word_ids[i-1] if i > 0 else True): # Only consider first token subword
        aligned_predictions.append(id2label[pred_id])
    elif word_ids[i] is None:
        continue # Skip specials: [CLS], [SEP], [PAD]
    else:
        continue # Skip subsequent subwords I-*


print("Original Sentence:", custom_test_sentence)
print("Predicted Tags:", aligned_predictions)

Original Sentence: 
Tim Cook to become
Apple Executive Chairman

John Ternus
to become Apple CEO

Predicted Tags: ['NNP', 'NNP', 'TO', 'VB', 'NNP', 'JJ', 'NN', 'NNP', 'NNP', 'TO', 'VB', 'NNP', 'NN']


---

## Task 7

text chuncking

In [ ]:
# build tag set and label mappings
all_tags = sorted({tag for s in train_sentences for tag in s['chunk_tags']})
label2id = {tag: i for i, tag in enumerate(all_tags)}
id2label = {i: tag for tag, i in label2id.items()}
num_labels = len(all_tags)
print('Tagset size:', num_labels)
print('Tags:', all_tags)
print(f"Tag IDs: {id2label}")
print(f"Label2ID: {label2id}")

Tagset size: 20
Tags: ['B-ADJP', 'B-ADVP', 'B-CONJP', 'B-INTJ', 'B-LST', 'B-NP', 'B-PP', 'B-PRT', 'B-SBAR', 'B-VP', 'I-ADJP', 'I-ADVP', 'I-CONJP', 'I-INTJ', 'I-LST', 'I-NP', 'I-PP', 'I-SBAR', 'I-VP', 'O']
Tag IDs: {0: 'B-ADJP', 1: 'B-ADVP', 2: 'B-CONJP', 3: 'B-INTJ', 4: 'B-LST', 5: 'B-NP', 6: 'B-PP', 7: 'B-PRT', 8: 'B-SBAR', 9: 'B-VP', 10: 'I-ADJP', 11: 'I-ADVP', 12: 'I-CONJP', 13: 'I-INTJ', 14: 'I-LST', 15: 'I-NP', 16: 'I-PP', 17: 'I-SBAR', 18: 'I-VP', 19: 'O'}
Label2ID: {'B-ADJP': 0, 'B-ADVP': 1, 'B-CONJP': 2, 'B-INTJ': 3, 'B-LST': 4, 'B-NP': 5, 'B-PP': 6, 'B-PRT': 7, 'B-SBAR': 8, 'B-VP': 9, 'I-ADJP': 10, 'I-ADVP': 11, 'I-CONJP': 12, 'I-INTJ': 13, 'I-LST': 14, 'I-NP': 15, 'I-PP': 16, 'I-SBAR': 17, 'I-VP': 18, 'O': 19}


In [ ]:
# map tokens and tags to token ids and label ids
def align_label(tokens, labels, sentence):
    """
    It maps:
      - O means the word doesn’t correspond to any entity.
      - B-PER/I-PER means the word corresponds to the beginning of/is inside a person entity.
      - B-ORG/I-ORG means the word corresponds to the beginning of/is inside an organization entity.
      - B-LOC/I-LOC means the word corresponds to the beginning of/is inside a location entity.
      - B-MISC/I-MISC means the word corresponds to the beginning of/is inside a miscellaneous entity.
    to each token in the sentence.

    When it runs out of tokens, it adds -100 to the label. Each word index that does not correspond to a word
    in the sentence, gets a -100. Each unknown token gets a -100.

    INPUTS:

    OUTPUT:
      - labels: 512 length of labels for each token in the sentence.
      - tokens: tokenizer output (BatchEncoding) with word_ids()
      - labels: list of original word-level labels for a single sentence

    INFO:
    512: is the model output size.
    """
    word_ids = tokens.word_ids()
    # print(f"Word IDS: {word_ids}")
    previous_word_idx = None
    label_ids = []
    # print(tokens)

    for word_idx in word_ids:
        # print(f"\n===\n\tword idx: {word_idx}")

        # Debug
        # if word_idx is not None:
        #     print(f"Sentence token: {sentence["tokens"][word_idx]}")
        #     print(f"Sentence label: {sentence["ner_tags"][word_idx]}")

        if word_idx is None:
            # special tokens
            label_ids.append(-100)
            # print(f"Special token: {label_ids}")

        elif word_idx != previous_word_idx:
            # first subword of a word
            # Get the corresponding token NER tag from the labels or -100
            label_ids.append(label2id.get(labels[word_idx], -100))
            # print(f"First subword: {label_ids}")

        else:
            # subsequent subword of the same word (ignore for loss)
            label_ids.append(-100)
            # print(f"Subsequent subword: {label_ids}")

        previous_word_idx = word_idx

    # print(f"Word IDX: {word_idx}, labels: {label_ids}")

    return label_ids

def encode(sentence):
    encodings = tokenizer(
        sentence['tokens'],
        truncation=True,
        padding='max_length',
        is_split_into_words=True,
        return_tensors='pt'  # get tensors directly
    )
    # print(f"Encodings: {encodings}")
    labels = align_label(encodings, sentence['chunk_tags'], sentence) # ner_tags -> chunk_tags
    # print(f"Final labels: {labels}")

    return {
        'input_ids': encodings['input_ids'].squeeze(0),        # [seq_len]
        'attention_mask': encodings['attention_mask'].squeeze(0),
        'labels': torch.tensor(labels, dtype=torch.long)
    }

print('encoding data')
# for sentence in train_sentences:
#     print(f"Sentence: {sentence}")
#     encode(sentence)
#     break
train_dataset = [encode(sentence) for sentence in train_sentences]
valid_dataset = [encode(sentence) for sentence in valid_sentences]
test_dataset = [encode(sentence) for sentence in test_sentences]

encoding data


In [ ]:
# Train and evalu model
Y_actual, Y_preds, y_true_tags, y_pred_tags = train_model(model, train_loader, valid_loader)
full_eval(Y_actual, Y_preds, y_true_tags, y_pred_tags)

training the model
epoch 1/3


Training epoch 1:   0%|          | 0/1756 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/407 [00:00<?, ?it/s]


=== Validation (epoch 1) — Token-level metrics ===
Accuracy          : 0.948
Balanced accuracy : 0.505

=== Validation (epoch 1) — Entity-level metrics (seqeval) ===
Precision : 0.902
Recall    : 0.902
F1        : 0.902
epoch 2/3


Training epoch 2:   0%|          | 0/1756 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/407 [00:00<?, ?it/s]


=== Validation (epoch 2) — Token-level metrics ===
Accuracy          : 0.954
Balanced accuracy : 0.579

=== Validation (epoch 2) — Entity-level metrics (seqeval) ===
Precision : 0.910
Recall    : 0.912
F1        : 0.911
epoch 3/3


Training epoch 3:   0%|          | 0/1756 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/407 [00:00<?, ?it/s]


=== Validation (epoch 3) — Token-level metrics ===
Accuracy          : 0.955
Balanced accuracy : 0.646

=== Validation (epoch 3) — Entity-level metrics (seqeval) ===
Precision : 0.912
Recall    : 0.914
F1        : 0.913

=== Test — Token-level classification report (sklearn) ===
              precision    recall  f1-score   support

      B-ADJP       0.85      0.56      0.68       306
      B-ADVP       0.83      0.80      0.81       650
     B-CONJP       0.80      0.40      0.53        10
      B-INTJ       0.00      0.00      0.00        31
       B-LST       0.20      0.33      0.25         3
        B-NP       0.96      0.97      0.96     14644
        B-PP       0.97      0.97      0.97      4884
       B-PRT       0.77      0.76      0.77       147
      B-SBAR       0.77      0.89      0.82       366
        B-VP       0.94      0.93      0.94      4696
      I-ADJP       0.53      0.60      0.56        52
      I-ADVP       0.77      0.51      0.62        39
     I-CONJP    

In [ ]:
# Find tst dataset text which fails chuncking
# Find a test sentence where the model fails to tag properly
for i in range(len(test_sentences)):
    sentence_tokens = test_sentences[i]['tokens']
    true_tags = y_true_tags[i]
    predicted_tags = y_pred_tags[i]

    if len(sentence_tokens) >= 13 and true_tags != predicted_tags:
        print(f"Sentence: {' '.join(sentence_tokens)}")
        print(f"True Tags: {true_tags}")
        print(f"Predicted Tags: {predicted_tags}")
        break

Sentence: Japan began the defence of their Asian Cup title with a lucky 2-1 win against Syria in a Group C championship match on Friday .
True Tags: ['B-NP', 'I-NP', 'B-PP', 'B-NP', 'O', 'B-ADVP', 'O', 'B-VP', 'I-VP', 'B-ADJP', 'B-PP', 'B-NP', 'I-NP', 'I-NP', 'O', 'B-NP', 'I-NP', 'I-NP', 'O', 'B-VP', 'B-PRT', 'B-PP', 'B-NP', 'B-SBAR', 'B-NP', 'B-VP', 'B-PRT', 'B-PP', 'B-NP', 'I-NP', 'B-PP', 'B-NP', 'I-NP', 'I-NP', 'B-PP', 'B-NP', 'O']
Predicted Tags: ['B-NP', 'I-NP', 'B-PP', 'B-NP', 'O', 'B-ADVP', 'O', 'B-VP', 'I-VP', 'B-ADJP', 'B-PP', 'B-NP', 'I-NP', 'B-NP', 'O', 'B-NP', 'I-NP', 'I-NP', 'O', 'B-VP', 'B-PRT', 'B-PP', 'B-NP', 'B-SBAR', 'B-NP', 'B-VP', 'B-PRT', 'B-PP', 'B-NP', 'I-NP', 'B-PP', 'B-NP', 'I-NP', 'I-NP', 'B-PP', 'B-NP', 'O']


In [ ]:
# Find a misclassified sentence - task 3 - should work for chuncks as well

find_the_errors_in_pos_tags(
    model,
    test_sentences,
    y_true_tags,
    y_pred_tags,
    tokenizer,
    id2label,
    device
    )

Misclassified sentence (over 13 words long!) at index 3:
Original words: Japan began the defence of their Asian Cup title with a lucky 2-1 win against Syria in a Group C championship match on Friday .
True Tags (aligned): ['B-NP', 'I-NP', 'B-PP', 'B-NP', 'O', 'B-ADVP', 'O', 'B-VP', 'I-VP', 'B-ADJP', 'B-PP', 'B-NP', 'I-NP', 'I-NP', 'O', 'B-NP', 'I-NP', 'I-NP', 'O', 'B-VP', 'B-PRT', 'B-PP', 'B-NP', 'B-SBAR', 'B-NP', 'B-VP', 'B-PRT', 'B-PP', 'B-NP', 'I-NP', 'B-PP', 'B-NP', 'I-NP', 'I-NP', 'B-PP', 'B-NP', 'O']
Predicted Tags (aligned): ['B-NP', 'I-NP', 'B-PP', 'B-NP', 'O', 'B-ADVP', 'O', 'B-VP', 'I-VP', 'B-ADJP', 'B-PP', 'B-NP', 'I-NP', 'B-NP', 'O', 'B-NP', 'I-NP', 'I-NP', 'O', 'B-VP', 'B-PRT', 'B-PP', 'B-NP', 'B-SBAR', 'B-NP', 'B-VP', 'B-PRT', 'B-PP', 'B-NP', 'I-NP', 'B-PP', 'B-NP', 'I-NP', 'I-NP', 'B-PP', 'B-NP', 'O']


{'correctly_classified_words': [('Japan', 'B-NP', 'B-NP'),
  ('began', 'I-NP', 'I-NP'),
  ('the', 'B-PP', 'B-PP'),
  ('defence', 'B-NP', 'B-NP'),
  ('of', 'O', 'O'),
  ('their', 'B-ADVP', 'B-ADVP'),
  ('Asian', 'O', 'O'),
  ('Cup', 'B-VP', 'B-VP'),
  ('title', 'I-VP', 'I-VP'),
  ('with', 'B-ADJP', 'B-ADJP'),
  ('a', 'B-PP', 'B-PP'),
  ('lucky', 'B-NP', 'B-NP'),
  ('2-1', 'I-NP', 'I-NP'),
  ('against', 'O', 'O'),
  ('Syria', 'B-NP', 'B-NP'),
  ('in', 'I-NP', 'I-NP'),
  ('a', 'I-NP', 'I-NP'),
  ('Group', 'O', 'O'),
  ('C', 'B-VP', 'B-VP'),
  ('championship', 'B-PRT', 'B-PRT'),
  ('match', 'B-PP', 'B-PP'),
  ('on', 'B-NP', 'B-NP'),
  ('Friday', 'B-SBAR', 'B-SBAR'),
  ('.', 'B-NP', 'B-NP')],
 'incorrectly_classified_words': [('win', 'I-NP', 'B-NP')]}

In [ ]:
# Run the test for the custom sentence
custom_test_sentence = """
Tim Cook to become
Apple Executive Chairman

John Ternus
to become Apple CEO
"""


# Tokenize
inputs_batch_encoding = tokenizer(
    custom_test_sentence.split(), # Split into words
    is_split_into_words=True,
    return_tensors='pt',
    padding='max_length',
    truncation=True
)

# Extract word_ids
word_ids = inputs_batch_encoding.word_ids()

# Move to device
inputs = {k: v.to(device) for k, v in inputs_batch_encoding.items()}

# Get preds
model.eval()
with torch.no_grad():
    outputs = model(**inputs)

# Get the predicted label IDs
logits = outputs.logits
predictions = torch.argmax(logits, dim=-1).squeeze().tolist()

# Map the predicted IDs to labels
aligned_predictions = []
for i, pred_id in enumerate(predictions):
    if word_ids[i] is not None and (word_ids[i] != word_ids[i-1] if i > 0 else True): # Only consider first token subword
        aligned_predictions.append(id2label[pred_id])
    elif word_ids[i] is None:
        continue # Skip specials: [CLS], [SEP], [PAD]
    else:
        continue # Skip subsequent subwords I-*


print("Original Sentence:", custom_test_sentence)
print("Predicted Tags:", aligned_predictions)

Original Sentence: 
Tim Cook to become
Apple Executive Chairman

John Ternus
to become Apple CEO

Predicted Tags: ['B-NP', 'I-NP', 'B-VP', 'I-VP', 'B-NP', 'I-NP', 'I-NP', 'I-NP', 'I-NP', 'B-VP', 'I-VP', 'B-NP', 'I-NP']


In [ ]:
print(f"Orig sentence: {custom_test_sentence.split()}")
print(f"Predicted tags: {aligned_predictions}")
print("Corrected Tags: ['B-NP', 'I-NP', 'B-VP', 'I-VP', 'B-NP', 'I-NP', 'I-NP', 'B-NP', 'I-NP', 'B-VP', 'I-VP', 'B-NP', 'I-NP']")

Orig sentence: ['Tim', 'Cook', 'to', 'become', 'Apple', 'Executive', 'Chairman', 'John', 'Ternus', 'to', 'become', 'Apple', 'CEO']
Predicted tags: ['B-NP', 'I-NP', 'B-VP', 'I-VP', 'B-NP', 'I-NP', 'I-NP', 'I-NP', 'I-NP', 'B-VP', 'I-VP', 'B-NP', 'I-NP']
Corrected Tags: ['B-NP', 'I-NP', 'B-VP', 'I-VP', 'B-NP', 'I-NP', 'I-NP', 'B-NP', 'I-NP', 'B-VP', 'I-VP', 'B-NP', 'I-NP']


---

## Task 8

In [8]:
!pip install transformers seqeval torch --quiet

In [9]:
from transformers import RobertaTokenizer, RobertaModel

tokenizer = RobertaTokenizer.from_pretrained(
    "roberta-base",
    add_prefix_space=True
    )
# model = RobertaModel.from_pretrained(
#     "roberta-base",
#     num_labels=num_labels,
#     id2label=id2label,
#     label2id=label2id
# )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
# Import the correct model for token classification
from transformers import RobertaForTokenClassification

In [11]:
# Load the model with classification head
model = RobertaForTokenClassification.from_pretrained(
    'roberta-base',
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForTokenClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.weight               | MISSING    | 
classifier.bias                 | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [14]:
# hyper-parameters
EPOCHS = 3
BATCH_SIZE = 8
LR = 1e-5

# use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)
optimizer = optim.AdamW(params=model.parameters(), lr=LR)

# prepare batches of data
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)
valid_loader = torch.utils.data.DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE
)
test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE
)

In [17]:
def report_metrics(Y_actual, Y_preds, y_true_tags, y_pred_tags, split_name):
    """Print both token-level and entity-level metrics."""
    print(f"\n=== {split_name} — Token-level metrics ===")
    print("Accuracy          : {:.3f}".format(accuracy_score(Y_actual, Y_preds)))
    print("Balanced accuracy : {:.3f}".format(balanced_accuracy_score(Y_actual, Y_preds)))

    print(f"\n=== {split_name} — Entity-level metrics (seqeval) ===")
    print("Precision : {:.3f}".format(seqeval_precision(y_true_tags, y_pred_tags)))
    print("Recall    : {:.3f}".format(seqeval_recall(y_true_tags, y_pred_tags)))
    print("F1        : {:.3f}".format(seqeval_f1(y_true_tags, y_pred_tags)))


# train the model
print('training the model')
for epoch in range(EPOCHS):
    model.train()
    print(f'epoch {epoch + 1}/{EPOCHS}')
    for batch in tqdm(train_loader, desc=f"Training epoch {epoch + 1}"):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # calculate performance on validation set
    Y_actual, Y_preds, y_true_tags, y_pred_tags = EvaluateModel(model, valid_loader)
    report_metrics(Y_actual, Y_preds, y_true_tags, y_pred_tags,
                   split_name=f"Validation (epoch {epoch + 1})")

print('\napplying the model to the test set')
Y_actual, Y_preds, y_true_tags, y_pred_tags = EvaluateModel(model, test_loader)

report_metrics(Y_actual, Y_preds, y_true_tags, y_pred_tags, split_name="Test")

training the model
epoch 1/3


Training epoch 1:   0%|          | 0/1756 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/407 [00:00<?, ?it/s]


=== Validation (epoch 1) — Token-level metrics ===
Accuracy          : 0.992
Balanced accuracy : 0.953

=== Validation (epoch 1) — Entity-level metrics (seqeval) ===
Precision : 0.946
Recall    : 0.956
F1        : 0.951
epoch 2/3


Training epoch 2:   0%|          | 0/1756 [00:00<?, ?it/s]

...

Training epoch 3:   0%|          | 0/1756 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/407 [00:00<?, ?it/s]


=== Validation (epoch 3) — Token-level metrics ===
Accuracy          : 0.991
Balanced accuracy : 0.961

=== Validation (epoch 3) — Entity-level metrics (seqeval) ===
Precision : 0.944
Recall    : 0.958
F1        : 0.951

applying the model to the test set


Evaluating:   0%|          | 0/432 [00:00<?, ?it/s]


=== Test — Token-level metrics ===
Accuracy          : 0.981
Balanced accuracy : 0.928

=== Test — Entity-level metrics (seqeval) ===
Precision : 0.897
Recall    : 0.926
F1        : 0.911


In [26]:
%%capture
model

In [20]:
full_eval(Y_actual, Y_preds, y_true_tags, y_pred_tags)


=== Test — Token-level classification report (sklearn) ===
              precision    recall  f1-score   support

       B-LOC       0.94      0.94      0.94      1668
      B-MISC       0.77      0.87      0.82       702
       B-ORG       0.89      0.93      0.91      1661
       B-PER       0.98      0.96      0.97      1617
       I-LOC       0.86      0.93      0.90       257
      I-MISC       0.51      0.79      0.62       216
       I-ORG       0.85      0.95      0.90       835
       I-PER       0.99      0.99      0.99      1156
           O       1.00      0.99      0.99     38323

    accuracy                           0.98     46435
   macro avg       0.87      0.93      0.89     46435
weighted avg       0.98      0.98      0.98     46435

=== Test — Entity-level classification report (seqeval) ===
              precision    recall  f1-score   support

         LOC      0.937     0.935     0.936      1668
        MISC      0.720     0.842     0.776       702
         ORG